# Crop Doctor — 20-class MobileNetV3-Large training (512 × 512) [Task R2]

This notebook runs the full retrain workflow in Google Colab under Task R2 rules:
- Dihedral perceptual hashing (64-bit canonical min across 8 rotations & flips).
- Cross-class conflict detection at Hamming distance <= 2 (excludes only true duplicates across different class folders).
- Within-class duplicate clustering (Hamming <= 8 default, <= 10 for augmented tomato classes); never cluster across classes.
- Grouped 80/20 cluster split: train on ALL photos of train clusters up to 640 per class; validate on ONE random photo per validation cluster (cap class total at 800).
- MobileNetV3-Large, ImageNet pretrained, 15 epochs, balanced sampling (WeightedRandomSampler) plus inverse-sqrt class weights.
- Platt temperature calibration, confidence threshold & free energy threshold.
- Pinned ExecuTorch 0.6.0 export matching app/build.gradle.kts.

**Important:** Resumable checkpoints are saved after every epoch to Google Drive. Run top-to-bottom.

## 1. Before starting

1. On your laptop, ZIP the **contents** of `C:\crop-data\crop_doctor_512` (the 20 class folders plus the two reports). Name it `crop_doctor_512.zip`.
2. Upload that ZIP to the top level of **Google Drive → My Drive**.
3. In Colab, choose **Runtime → Change runtime type → T4 GPU** if offered. Save.
4. Run the cells in order. If Colab gives a different GPU, the notebook prints its name.

In [ ]:
# Detect environment: Kaggle or Google Colab
import os, sys
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print("✓ Detected Environment: Kaggle (GPU Accelerator)")
elif IS_COLAB:
    print("✓ Detected Environment: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Detected Environment: Generic / Local")


In [ ]:
import concurrent.futures
import os, sys, json, zipfile, random, hashlib, shutil, math, time, platform
from pathlib import Path
from collections import defaultdict, Counter

SEED = 42
MAX_TRAIN_PER_CLASS = 640
MAX_CLASS_TOTAL = 800
HAMMING_DIST_DEFAULT = 8
HAMMING_DIST_TOMATO = 10
CONFLICT_MAX_DISTANCE = 2
IMAGE_SIZE = 512
EPOCHS = 15
BATCH_SIZE = 8
NUM_WORKERS = 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TARGET_VALIDATION_PRECISION = 0.90

CLASS_NAMES = [
    "rice_bacterial_leaf_blight",
    "rice_leaf_blast",
    "rice_sheath_blight",
    "rice_healthy",
    "cotton_bacterial_blight",
    "cotton_fusarium_wilt",
    "cotton_leaf_curl_virus",
    "cotton_healthy",
    "chilli_leaf_curl_virus",
    "chilli_nutrition_deficiency",
    "chilli_healthy",
    "tomato_leaf_blight",
    "tomato_yellow_leaf_curl_virus",
    "tomato_septoria_leaf_spot",
    "tomato_healthy",
    "maize_gray_leaf_spot",
    "maize_common_rust",
    "maize_northern_leaf_blight",
    "maize_healthy",
    "weeds_not_a_crop",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".jfif"}

# Dynamic workspace paths
if IS_KAGGLE:
    RUN_DIR = Path("/kaggle/working/runs_final")
    EXTRACT_DIR = Path("/kaggle/working/crop_doctor_512_extracted")
    kaggle_zips = list(Path("/kaggle/input").rglob("crop_doctor_512.zip")) + list(Path("/kaggle/input").rglob("*.zip"))
    DRIVE_ZIP = str(kaggle_zips[0]) if kaggle_zips else "/kaggle/input/crop_doctor_512.zip"
else:
    DRIVE_ZIP = "/content/drive/MyDrive/crop_doctor_512.zip"
    EXTRACT_DIR = Path("/content/crop_doctor_512_extracted")
    RUN_DIR = Path("/content/drive/MyDrive/Crop_Doctor_Training/runs_final")

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ARTIFACT_DIR = RUN_DIR / "artifacts"
for p in [RUN_DIR, CHECKPOINT_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except Exception:
    pass

print("Python:", sys.version)
print("Dataset source path:", DRIVE_ZIP)
print("Run directory:", RUN_DIR)


In [ ]:
# Install the small image-hashing dependency. PyTorch / torchvision are supplied by Colab.
%pip -q install ImageHash

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile
import imagehash
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, precision_recall_curve)
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = False
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Locate or extract dataset (Kaggle pre-extracted or Colab/Kaggle ZIP with \ normalization)
DATA_ROOT = None

# 1. If running on Kaggle and dataset was uploaded uncompressed:
if IS_KAGGLE:
    search_roots = [Path("/kaggle/input")] + [p for p in Path("/kaggle/input").rglob("*") if p.is_dir()]
    for candidate in search_roots:
        if all((candidate / c).is_dir() for c in CLASS_NAMES):
            DATA_ROOT = candidate
            print(f"✓ Found pre-extracted dataset directly in Kaggle input: {DATA_ROOT}")
            break

# 2. Extract from ZIP if not already pre-extracted:
if DATA_ROOT is None:
    if not Path(DRIVE_ZIP).exists():
        if IS_KAGGLE:
            raise FileNotFoundError("Dataset ZIP not found in /kaggle/input!\n" \
                                    "Click '+ Add Input' on the right sidebar, upload 'crop_doctor_512.zip', and re-run.")
        else:
            raise FileNotFoundError(f"Upload crop_doctor_512.zip to My Drive, then confirm DRIVE_ZIP: {DRIVE_ZIP}")
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Extracting dataset ZIP from {DRIVE_ZIP} (handling Windows \\)...\n")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        for member in zf.infolist():
            norm_name = member.filename.replace("\\", "/")
            target_path = EXTRACT_DIR / norm_name
            if member.is_dir() or norm_name.endswith("/"):
                target_path.mkdir(parents=True, exist_ok=True)
            else:
                target_path.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src_file, open(target_path, "wb") as dst_file:
                    shutil.copyfileobj(src_file, dst_file)

    candidates = [EXTRACT_DIR] + [p for p in EXTRACT_DIR.rglob("*") if p.is_dir()]
    for candidate in candidates:
        if all((candidate / c).is_dir() for c in CLASS_NAMES):
            DATA_ROOT = candidate
            break

if DATA_ROOT is None:
    visible = [p.name for p in EXTRACT_DIR.iterdir()] if EXTRACT_DIR.exists() else []
    raise RuntimeError("Could not find all 20 class folders. Extracted top-level entries: " + str(visible))

print("✓ Dataset root identified successfully:", DATA_ROOT)
for c in CLASS_NAMES:
    count = len(list((DATA_ROOT / c).iterdir()))
    print(f"  {c}: {count} files")


In [ ]:
# Inventory and image integrity check: every source photo must be readable and exactly 512x512.
import concurrent.futures

records = []
invalid = []

def check_image(item):
    p, class_name = item
    try:
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im)
            width, height = im.size
            mode = im.mode
            if (width, height) != (512, 512):
                return None, {"path": str(p), "reason": f"dimension_{width}x{height}"}
            im.convert("RGB").load()
        return {
            "path": str(p),
            "relative_path": str(p.relative_to(DATA_ROOT)),
            "class_name": class_name,
            "class_index": CLASS_TO_IDX[class_name],
            "width": width,
            "height": height,
            "mode": mode,
        }, None
    except Exception as e:
        return None, {"path": str(p), "reason": repr(e)}

all_candidates = []
for class_name in CLASS_NAMES:
    class_dir = DATA_ROOT / class_name
    files = sorted([p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    for p in files:
        all_candidates.append((p, class_name))

print(f"Checking integrity of {len(all_candidates):,} images (multithreaded)...")
t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    for i, (rec, inv) in enumerate(ex.map(check_image, all_candidates), 1):
        if rec: records.append(rec)
        if inv: invalid.append(inv)
        if i % 2500 == 0 or i == len(all_candidates):
            print(f"  [Checked {i:,} / {len(all_candidates):,} images ({time.time()-t0:.1f}s)]")

raw_counts = {c: sum(r["class_name"] == c for r in records) for c in CLASS_NAMES}
print("Readable 512x512 images:", len(records))
print("Invalid / wrong-size images:", len(invalid))
print(pd.DataFrame([{"class": c, "images": raw_counts[c]} for c in CLASS_NAMES]).to_string(index=False))
if any(raw_counts[c] == 0 for c in CLASS_NAMES):
    raise RuntimeError("At least one class has no valid images. Fix the dataset before training.")
with open(ARTIFACT_DIR / "preprocessing_validation.json", "w") as f:
    json.dump({"image_size": [512,512], "readable_images": len(records), "invalid": invalid, "class_counts": raw_counts}, f, indent=2)


## 2. Duplicate clustering & Cross-Class Conflict Exclusion [Task R2]

Task R2 build rules:
1. **Canonical Dihedral pHash**: Computes 64-bit perceptual hash across 8 dihedral transformations (original, rot90, rot180, rot270, and horizontal mirrors). Uses the canonical minimum hash.
2. **Cross-Class Conflict Check**: A photo is conflicting ONLY if the exact same photo appears in two different class folders at Hamming distance <= 2. Excludes only those files.
3. **Within-Class Clustering**: Cluster WITHIN each class independently (Hamming <= 8 for default classes, <= 10 for augmented tomato classes). Never cluster across classes.
4. **Grouped 80/20 Cluster Split**: Split whole clusters 80/20. Train on ALL photos of train clusters up to 640 per class. Validate on ONE random photo per validation cluster to prevent accuracy inflation.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.rank[ra] < self.rank[rb]: ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]: self.rank[ra] += 1

def compute_canonical_dihedral_phash(img_path):
    with Image.open(img_path) as im0:
        im = ImageOps.exif_transpose(im0).convert('RGB')
        m = ImageOps.mirror(im)
        variants = [
            im,
            im.rotate(90, expand=True),
            im.rotate(180, expand=True),
            im.rotate(270, expand=True),
            m,
            m.rotate(90, expand=True),
            m.rotate(180, expand=True),
            m.rotate(270, expand=True),
        ]
        hashes = [imagehash.phash(v) for v in variants]
        return min(hashes, key=lambda h: h.hash.tobytes())

print(f"Computing canonical dihedral pHashes for {len(records):,} images (multithreaded)...")
t0_hash = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    for i, (rec, h) in enumerate(zip(records, ex.map(lambda r: compute_canonical_dihedral_phash(r['path']), records)), 1):
        rec['phash'] = h
        if i % 3000 == 0 or i == len(records):
            print(f"  [Hashed {i:,} / {len(records):,} images ({time.time()-t0_hash:.1f}s)]")

# Rule 2(c): Cross-class conflicting pairs (Hamming distance <= 2 across DIFFERENT classes)
class_to_records = defaultdict(list)
for r in records:
    class_to_records[r["class_name"]].append(r)

conflicting_paths = set()
conflict_examples = defaultdict(list)
class_list = sorted(class_to_records.keys())

for i in range(len(class_list)):
    c_a = class_list[i]
    for j in range(i + 1, len(class_list)):
        c_b = class_list[j]
        for r_a in class_to_records[c_a]:
            for r_b in class_to_records[c_b]:
                dist = r_a["phash"] - r_b["phash"]
                if dist <= CONFLICT_MAX_DISTANCE:
                    conflicting_paths.add(r_a["path"])
                    conflicting_paths.add(r_b["path"])
                    if len(conflict_examples[c_a]) < 10:
                        conflict_examples[c_a].append({"this": r_a["relative_path"], "other": r_b["relative_path"], "other_class": c_b, "dist": dist})
                    if len(conflict_examples[c_b]) < 10:
                        conflict_examples[c_b].append({"this": r_b["relative_path"], "other": r_a["relative_path"], "other_class": c_a, "dist": dist})

print(f"Total conflicting photos excluded (Hamming <= {CONFLICT_MAX_DISTANCE} cross-class): {len(conflicting_paths)}")

# Rule 2(a) & 2(b): Cluster WITHIN each class only
class_clusters = {}
class_cluster_counts = {}
for c in CLASS_NAMES:
    c_records = [r for r in class_to_records[c] if r["path"] not in conflicting_paths]
    n = len(c_records)
    uf = UnionFind(n)
    threshold = HAMMING_DIST_TOMATO if c.startswith("tomato_") else HAMMING_DIST_DEFAULT
    for idx_a in range(n):
        for idx_b in range(idx_a + 1, n):
            if c_records[idx_a]["phash"] - c_records[idx_b]["phash"] <= threshold:
                uf.union(idx_a, idx_b)
    clusters = defaultdict(list)
    for idx in range(n):
        root = uf.find(idx)
        clusters[root].append(c_records[idx])
    class_clusters[c] = list(clusters.values())
    class_cluster_counts[c] = len(class_clusters[c])
    print(f"Class {c:30s} | records: {len(c_records):4d} | clusters: {len(class_clusters[c]):4d} (threshold: {threshold})")

duplicate_report = {
    "conflicting_count": len(conflicting_paths),
    "conflict_examples": {k: list(v) for k, v in conflict_examples.items()},
    "clusters_per_class": class_cluster_counts
}
with open(ARTIFACT_DIR / "duplicate_report.json", "w") as f:
    json.dump(duplicate_report, f, indent=2)


In [ ]:
# Rule 2(d): Split clusters 80/20. Train on ALL photos of train clusters up to 640 per class.
# Validate on ONE random photo per validation cluster. Cap total at 800.
# Rule 2(e): Report per class: source files, clusters, removed same-label duplicates,
# conflicting files, train photos, validation photos. Mark SMALL if < 150 clusters or < 30 val photos.

rng = random.Random(SEED)
train_items, val_items = [], []
class_counts_v2 = {}
cluster_counter = 0

for c in CLASS_NAMES:
    clusters = list(class_clusters[c])
    cluster_count = len(clusters)
    source_n = raw_counts[c]
    conflict_n = sum(1 for r in class_to_records[c] if r['path'] in conflicting_paths)

    # Assign unique cluster_id across dataset
    for cl in clusters:
        cid = f"{c}_cluster_{cluster_counter:05d}"
        cluster_counter += 1
        for item in cl:
            item["cluster_id"] = cid

    # Shuffle clusters
    rng.shuffle(clusters)
    if cluster_count >= 2:
        n_val = max(1, int(round(0.20 * cluster_count)))
        n_val = min(n_val, cluster_count - 1)
    else:
        n_val = 0

    val_clusters = clusters[:n_val]
    train_clusters = clusters[n_val:]

    # Train on ALL photos of train clusters up to 640
    train_photos_pool = [p for cl in train_clusters for p in cl]
    rng.shuffle(train_photos_pool)
    train_class = train_photos_pool[:MAX_TRAIN_PER_CLASS]

    # Validate on ONE random photo per validation cluster
    val_class = [rng.choice(cl) for cl in val_clusters]

    train_items.extend(train_class)
    val_items.extend(val_class)

    train_n = len(train_class)
    val_n = len(val_class)
    kept_n = train_n + val_n
    same_dup_n = source_n - conflict_n - kept_n
    is_small = (cluster_count < 150) or (val_n < 30)

    class_counts_v2[c] = {
        "source_valid_512_count": source_n,
        "unique_clusters": cluster_count,
        "selected_unique_images_after_cap": kept_n,
        "train_count": train_n,
        "validation_count": val_n,
        "same_label_duplicate_files_removed": same_dup_n,
        "conflicting_label_files_excluded": conflict_n,
        "small_class": is_small,
        "cap": MAX_CLASS_TOTAL,
        "train_cap": MAX_TRAIN_PER_CLASS,
    }

rng.shuffle(train_items)
rng.shuffle(val_items)
train_groups = {x["cluster_id"] for x in train_items}
val_groups = {x["cluster_id"] for x in val_items}
overlap = sorted(train_groups & val_groups)
leakage = {
    "train_samples": len(train_items),
    "validation_samples": len(val_items),
    "train_cluster_count": len(train_groups),
    "validation_cluster_count": len(val_groups),
    "overlapping_cluster_count": len(overlap),
    "overlapping_cluster_ids": overlap,
    "zero_cluster_leakage": len(overlap) == 0,
}

with open(ARTIFACT_DIR / "class_counts_v2.json", "w") as f:
    json.dump(class_counts_v2, f, indent=2)
with open(ARTIFACT_DIR / "split_v2.json", "w") as f:
    json.dump({"seed": SEED, "method": "within-class perceptual-hash cluster grouping; 80/20 split; max 640 train, 1 val per cluster",
               "train": train_items, "validation": val_items}, f, indent=2, default=str)
with open(ARTIFACT_DIR / "leakage_after.json", "w") as f:
    json.dump(leakage, f, indent=2)

# One deterministic validation probe image per class
probe = []
for c in CLASS_NAMES:
    options = [x for x in val_items if x["class_name"] == c]
    if options:
        probe.append(sorted(options, key=lambda x: x["relative_path"])[0])
with open(ARTIFACT_DIR / "probe_set.json", "w") as f:
    json.dump({"purpose": "fixed validation probe images; not used for training", "samples": probe}, f, indent=2, default=str)

print(f"Train samples: {len(train_items)}, Validation samples: {len(val_items)}")
print("Cluster leakage between train and val:", len(overlap))
print("\n" + "="*105)
print(f"{'Class':<30} | {'Source':<7} | {'Clusters':<8} | {'SameDup':<7} | {'Conflict':<8} | {'Train':<6} | {'Val':<5} | {'Kept':<6} | {'Status'}")
print("="*105)
for c in CLASS_NAMES:
    s = class_counts_v2[c]
    st = "SMALL" if s["small_class"] else "OK"
    print(f"{c:<30} | {s['source_valid_512_count']:<7} | {s['unique_clusters']:<8} | {s['same_label_duplicate_files_removed']:<7} | {s['conflicting_label_files_excluded']:<8} | {s['train_count']:<6} | {s['validation_count']:<5} | {s['selected_unique_images_after_cap']:<6} | {st}")
print("="*105)
if overlap:
    raise RuntimeError("Leakage detected: a cluster appears in both train and validation.")

# User requirement: no image class should drop below 300 images
MIN_CLASS_TOTAL = 300
for c in CLASS_NAMES:
    total_kept = class_counts_v2[c]["selected_unique_images_after_cap"]
    if total_kept < MIN_CLASS_TOTAL:
        raise RuntimeError(f"Class {c} dropped below required minimum of {MIN_CLASS_TOTAL} images! (Has {total_kept})")
lowest_count = min(class_counts_v2[c]["selected_unique_images_after_cap"] for c in CLASS_NAMES)
print(f"\n✓ Verified: All 20 classes have at least {MIN_CLASS_TOTAL} images (lowest is {lowest_count} images).\n")


## 3. Train MobileNetV3-Large

The notebook uses inverse-square-root class weights in the loss (not an additional weighted sampler). Checkpoints and the best model are saved to Google Drive after every epoch so the run can be resumed after a disconnect.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(512, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(20, fill=(114, 114, 114)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

from torch.utils.data import WeightedRandomSampler
class CropDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        row = self.items[idx]
        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        return self.transform(image), int(row["class_index"])

train_ds = CropDataset(train_items, train_transform)
val_ds = CropDataset(val_items, val_transform)

# Rule 4: Balanced sampling via WeightedRandomSampler
train_targets = [int(item["class_index"]) for item in train_items]
train_counts_counter = Counter(train_targets)
sample_weights = [1.0 / train_counts_counter[t] for t in train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_items), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# ImageNet pretrained MobileNetV3-Large, replaced with exactly 20 output logits
weights = models.MobileNet_V3_Large_Weights.DEFAULT
model = models.mobilenet_v3_large(weights=weights)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASS_NAMES))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Inverse-sqrt class weights (Rule 4)
counts = torch.tensor([max(1, class_counts_v2[c]["train_count"]) for c in CLASS_NAMES], dtype=torch.float32)
class_weights = (1.0 / torch.sqrt(counts))
class_weights = class_weights / class_weights.mean()
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print("Device:", device)
print("Model output classes:", model.classifier[3].out_features)
print("Input shape:", (1, 3, 512, 512))
print("Train batches:", len(train_loader), "Validation batches:", len(val_loader))


In [ ]:
# Resume from the latest Drive checkpoint if it exists.
LATEST_CKPT = CHECKPOINT_DIR / "latest_checkpoint.pth"
BEST_MODEL_PATH = ARTIFACT_DIR / "best_model.pth"
HISTORY_PATH = ARTIFACT_DIR / "training_history.json"

start_epoch = 0
best_val_loss = float("inf")
history = []
if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    start_epoch = int(ckpt["epoch"]) + 1
    best_val_loss = float(ckpt.get("best_val_loss", best_val_loss))
    history = ckpt.get("history", [])
    print("Resuming from epoch", start_epoch + 1)
else:
    print("Starting new training run.")

def run_epoch(loader, training, epoch_num=1):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    t_start = time.time()
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        if training and ((batch_idx + 1) % 250 == 0 or (batch_idx + 1) == len(loader)):
            elapsed = time.time() - t_start
            print(f"  [Epoch {epoch_num:02d}/{EPOCHS} | Batch {batch_idx+1:04d}/{len(loader)}] running loss: {total_loss/total:.4f} acc: {correct/total:.4f} ({elapsed:.1f}s)")
    return total_loss / max(total, 1), correct / max(total, 1)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, True, epoch + 1)
    val_loss, val_acc = run_epoch(val_loader, False, epoch + 1)
    scheduler.step()
    row = {"epoch": epoch + 1, "train_loss": train_loss, "train_accuracy": train_acc,
           "validation_loss": val_loss, "validation_accuracy": val_acc,
           "learning_rate": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0}
    history.append(row)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model_state": model.state_dict(), "class_names": CLASS_NAMES,
                    "input_size": 512, "best_val_loss": best_val_loss}, BEST_MODEL_PATH)
    torch.save({"epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_val_loss": best_val_loss, "history": history}, LATEST_CKPT)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} COMPLETE | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f} | {row['seconds']:.1f}s\n")

print("Training loop finished. Best checkpoint:", BEST_MODEL_PATH)


In [ ]:
# Load best checkpoint and collect validation logits.
best = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(best["model_state"])
model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(images)
        all_logits.append(logits.float().cpu())
        all_labels.append(labels.cpu())
val_logits = torch.cat(all_logits, dim=0)
val_labels = torch.cat(all_labels, dim=0)
val_probs = torch.softmax(val_logits, dim=1).numpy()
y_true = val_labels.numpy()
y_pred = val_probs.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
                               target_names=CLASS_NAMES, output_dict=True, zero_division=0)
per_class_accuracy = {}
for i, c in enumerate(CLASS_NAMES):
    support = int((y_true == i).sum())
    correct = int(((y_true == i) & (y_pred == i)).sum())
    per_class_accuracy[c] = {"accuracy_recall": (correct / support if support else None),
                             "correct": correct, "support": support,
                             "precision": report[c]["precision"],
                             "recall": report[c]["recall"],
                             "f1_score": report[c]["f1-score"]}

overall_acc = float(accuracy_score(y_true, y_pred))
macro_acc = float(balanced_accuracy_score(y_true, y_pred))

# Confusion matrix image + CSV
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(ARTIFACT_DIR / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(17, 15))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Crop Doctor — 20-class confusion matrix")
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=180)
plt.show()

# Top off-diagonal confusions
off = []
for i, true_name in enumerate(CLASS_NAMES):
    for j, pred_name in enumerate(CLASS_NAMES):
        if i != j and cm[i, j] > 0:
            off.append({"true_class": true_name, "predicted_class": pred_name, "count": int(cm[i, j])})
off = sorted(off, key=lambda x: x["count"], reverse=True)[:10]

metrics = {
    "model": "MobileNetV3-Large",
    "input_shape": [1, 3, 512, 512],
    "output_shape": [1, 20],
    "class_order": CLASS_NAMES,
    "train_images": len(train_items),
    "validation_images": len(val_items),
    "epochs_completed": len(history),
    "overall_accuracy": overall_acc,
    "macro_accuracy_balanced_accuracy": macro_acc,
    "per_class_grouped_accuracy": per_class_accuracy,
    "top_confusions": off,
    "calibration_method": "pending temperature scaling cell",
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "torchvision": torchvision.__version__, "executorch": None},
}
with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Rule 4: Report per-class accuracy with the number of validation photos next to each number, and macro accuracy
print("\n" + "="*75)
print(f"{'Class':<32} | {'Val Accuracy':<14} | {'Correct':<8} | {'Val Photos'}")
print("="*75)
for c in CLASS_NAMES:
    pca = per_class_accuracy[c]
    acc_str = f"{pca['accuracy_recall']*100:.2f}%" if pca['accuracy_recall'] is not None else "N/A"
    print(f"{c:<32} | {acc_str:<14} | {pca['correct']:<8} | {pca['support']}")
print("="*75)
print(f"Macro Accuracy (Balanced): {macro_acc*100:.2f}%")
print(f"Overall Accuracy:          {overall_acc*100:.2f}%")
print("Top confusions:", off[:5])


In [ ]:
# Fit one scalar temperature on validation logits.
# Also save in thresholds.json: temperature, confidence_threshold and energy_threshold with its method (Rule 5)
import torch.nn.functional as F
logits_cpu = val_logits.detach().float()
labels_cpu = val_labels.detach().long()
log_temperature = nn.Parameter(torch.zeros(1))
cal_optimizer = torch.optim.LBFGS([log_temperature], lr=0.05, max_iter=100)

def calibration_closure():
    cal_optimizer.zero_grad()
    temp = torch.exp(log_temperature).clamp(0.05, 20.0)
    loss = F.cross_entropy(logits_cpu / temp, labels_cpu)
    loss.backward()
    return loss

nll_before = float(F.cross_entropy(logits_cpu, labels_cpu).item())
cal_optimizer.step(calibration_closure)
temperature = float(torch.exp(log_temperature).clamp(0.05, 20.0).item())
calibrated_probs = torch.softmax(logits_cpu / temperature, dim=1).numpy()
confidence = calibrated_probs.max(axis=1)
cal_pred = calibrated_probs.argmax(axis=1)
is_correct = (cal_pred == y_true).astype(int)
precision, recall, thresholds = precision_recall_curve(is_correct, confidence)
valid_ix = [i for i, th in enumerate(thresholds) if precision[i] >= TARGET_VALIDATION_PRECISION]
if valid_ix:
    best_ix = max(valid_ix, key=lambda i: recall[i])
    confidence_threshold = float(thresholds[best_ix])
    threshold_note = "lowest-confidence operating point meeting target precision on validation data"
    target_met = True
else:
    confidence_threshold = 1.0
    threshold_note = "target precision not reached on validation data; threshold set to 1.0 placeholder"
    target_met = False

nll_after = float(F.cross_entropy(logits_cpu / temperature, labels_cpu).item())

# Compute free energy threshold: E(x; T) = -T * ln(sum(exp(z_i / T)))
# In-distribution validation energy 95th percentile coverage
scaled_logits = logits_cpu / temperature
free_energies = -temperature * torch.logsumexp(scaled_logits, dim=1).numpy()
energy_threshold = float(np.percentile(free_energies, 95))
energy_method = "free_energy_validation_95th_percentile"

thresholds_obj = {
    "calibration_method": "single_scalar_temperature_scaling",
    "temperature": temperature,
    "confidence_threshold": confidence_threshold,
    "default_threshold": confidence_threshold,
    "energy_threshold": energy_threshold,
    "energy_method": energy_method,
    "nll_before": nll_before,
    "nll_after": nll_after,
    "threshold_target_precision": TARGET_VALIDATION_PRECISION,
    "target_precision_reached": target_met,
    "selection_note": threshold_note,
    "status": "provisional; calibrated on Task R2 grouped validation split",
}
with open(ARTIFACT_DIR / "thresholds.json", "w") as f:
    json.dump(thresholds_obj, f, indent=2)

metrics["calibration_method"] = "single_scalar_temperature_scaling"
metrics["temperature"] = temperature
metrics["confidence_threshold"] = confidence_threshold
metrics["energy_threshold"] = energy_threshold
metrics["energy_method"] = energy_method
metrics["validation_nll_before_temperature_scaling"] = nll_before
metrics["validation_nll_after_temperature_scaling"] = nll_after
with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(thresholds_obj, indent=2))


## 4. Export to ExecuTorch (`vision.pte`) and parity test

The export targets **XNNPACK CPU**, which is a standard ExecuTorch backend for Android CPU deployment. The current official ExecuTorch workflow uses `torch.export.export` followed by `to_edge_transform_and_lower` and `XnnpackPartitioner`. The `.pte` size must be measured; the notebook will report whether it is under 30 MB. The Android app's ExecuTorch runtime version must be compatible with the version used here, so record the version in `metrics.json` and have the integration step align the Android dependency to it.

In [ ]:
# Pin ExecuTorch to 0.6.0 to match org.pytorch:executorch-android:0.6.0 in app/build.gradle.kts (Rule 5)
APP_REQUIRED_EXECUTORCH = "0.6.0"

if sys.version_info >= (3, 13):
    print(f"Detected Python {sys.version.split()[0]} (>= 3.13).")
    print("Setting up automated Python 3.11 environment for ExecuTorch 0.6.0 export via uv...")
    !pip install -q uv
    !uv venv --python 3.11 /tmp/py311
    !uv pip install --python /tmp/py311 --no-deps "https://download.pytorch.org/whl/cpu/executorch-0.6.0%2Bcpu-cp311-cp311-manylinux_2_28_x86_64.whl"
    !uv pip install --python /tmp/py311 ruamel.yaml tabulate packaging sympy flatbuffers
    !uv pip install --python /tmp/py311 --extra-index-url https://download.pytorch.org/whl/cpu torch torchvision
    EXPORT_PYTHON = "/tmp/py311/bin/python"
    installed_et = "0.6.0+cpu"
else:
    %pip -q install executorch==0.6.0
    import importlib.metadata as importlib_metadata
    try:
        installed_et = importlib_metadata.version("executorch")
    except Exception:
        installed_et = "unknown"
    EXPORT_PYTHON = sys.executable

print(f"ExecuTorch export environment ready: {installed_et} (App requires: {APP_REQUIRED_EXECUTORCH})")
if not installed_et.startswith(APP_REQUIRED_EXECUTORCH):
    raise RuntimeError(f"STOP: ExecuTorch version mismatch! Installed {installed_et}, but app requires {APP_REQUIRED_EXECUTORCH}.")


In [ ]:
# Reload best model weights and export to ExecuTorch XNNPACK (.pte)
export_script = f'''
import sys, json
from pathlib import Path
import torch
from torch import nn
from torchvision import models

RUN_DIR = Path("{str(RUN_DIR)}")
ARTIFACT_DIR = Path("{str(ARTIFACT_DIR)}")
BEST_MODEL_PATH = Path("{str(BEST_MODEL_PATH)}")
PTE_PATH = ARTIFACT_DIR / "vision.pte"
CLASS_NAMES = {CLASS_NAMES}

print("Exporting vision.pte with Python:", sys.version)
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

model_export = models.mobilenet_v3_large(weights=None)
model_export.classifier[3] = nn.Linear(model_export.classifier[3].in_features, len(CLASS_NAMES))
ckpt = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model_export.load_state_dict(ckpt["model_state"])
model_export.eval()

example = torch.randn(1, 3, 512, 512, dtype=torch.float32)
exported_program = torch.export.export(model_export, (example,))
edge_program = to_edge_transform_and_lower(exported_program, partitioner=[XnnpackPartitioner()])
et_program = edge_program.to_executorch()

with open(PTE_PATH, "wb") as f:
    f.write(et_program.buffer)

pte_mb = PTE_PATH.stat().st_size / (1024 * 1024)
print(f"✓ Successfully exported vision.pte: {pte_mb:.2f} MB (Rule H15 budget < 30 MB: {pte_mb < 30.0})")
'''

export_py_path = RUN_DIR / "export_xnnpack.py"
with open(export_py_path, "w") as f:
    f.write(export_script)

!{EXPORT_PYTHON} {export_py_path}

PTE_PATH = ARTIFACT_DIR / "vision.pte"
assert PTE_PATH.exists(), "vision.pte failed to export!"
pte_mb = PTE_PATH.stat().st_size / (1024 * 1024)
print(f"Verified PTE file size: {pte_mb:.2f} MB")


In [ ]:
# Parity test between PyTorch model and ExecuTorch runtime on validation samples
parity_script = f'''
import sys, json, random
from pathlib import Path
from PIL import Image, ImageOps
import torch
from torchvision import models, transforms

RUN_DIR = Path("{str(RUN_DIR)}")
ARTIFACT_DIR = Path("{str(ARTIFACT_DIR)}")
BEST_MODEL_PATH = Path("{str(BEST_MODEL_PATH)}")
PTE_PATH = ARTIFACT_DIR / "vision.pte"
CLASS_NAMES = {CLASS_NAMES}
SEED = {SEED}

from executorch.runtime import Runtime
runtime = Runtime.get()
et_program_runtime = runtime.load_program(str(PTE_PATH))
et_method = et_program_runtime.load_method("forward")

model_export = models.mobilenet_v3_large(weights=None)
model_export.classifier[3] = torch.nn.Linear(model_export.classifier[3].in_features, len(CLASS_NAMES))
ckpt = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model_export.load_state_dict(ckpt["model_state"])
model_cpu = model_export.eval()

with open(ARTIFACT_DIR / "split_v2.json") as f:
    val_items = json.load(f)["val_items"]

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

parity_rows = random.Random(SEED).sample(val_items, k=min(10, len(val_items)))
parity_results = []
for row in parity_rows:
    with Image.open(row["path"]) as im:
        image = ImageOps.exif_transpose(im).convert("RGB")
    x = val_transform(image).unsqueeze(0).float()
    with torch.no_grad():
        torch_out = model_cpu(x).cpu()
    et_result = et_method.execute([x.contiguous()])
    et_out = et_result[0] if isinstance(et_result, (list, tuple)) else et_result
    if not isinstance(et_out, torch.Tensor):
        et_out = torch.tensor(et_out)
    et_out = et_out.detach().cpu().float()
    abs_diff = (torch_out - et_out).abs()
    max_abs = float(abs_diff.max().item())
    mean_abs = float(abs_diff.mean().item())
    parity_results.append({{
        "relative_path": row["relative_path"],
        "true_class": row["class_name"],
        "max_absolute_error": max_abs,
        "mean_absolute_error": mean_abs,
        "pass_tolerance_1e-3": max_abs <= 1e-3,
    }})

parity_obj = {{
    "backend": "XNNPACK",
    "input_shape": [1, 3, 512, 512],
    "output_shape": [1, 20],
    "tolerance_max_absolute_error": 1e-3,
    "samples_tested": len(parity_results),
    "all_pass": all(x["pass_tolerance_1e-3"] for x in parity_results),
    "results": parity_results,
}}
with open(ARTIFACT_DIR / "parity_samples.json", "w") as f:
    json.dump(parity_obj, f, indent=2)

print(f"Parity verification: {{parity_obj['samples_tested']}} samples tested, all_pass={{parity_obj['all_pass']}}")
'''

parity_py_path = RUN_DIR / "verify_parity.py"
with open(parity_py_path, "w") as f:
    f.write(parity_script)

!{EXPORT_PYTHON} {parity_py_path}

with open(ARTIFACT_DIR / "parity_samples.json") as f:
    parity_obj = json.load(f)
print("✓ Parity verification passed:", parity_obj["all_pass"])


In [ ]:
# Final labels file uses the exact index order. Update metrics with actual export versions/size.
with open(ARTIFACT_DIR / "labels.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

if "versions" not in metrics:
    metrics["versions"] = {}
metrics["versions"]["executorch"] = "0.6.0"
metrics["versions"]["torch"] = torch.__version__
metrics["versions"]["torchvision"] = torchvision.__version__
metrics["versions"]["python"] = platform.python_version()
metrics["model_file_size_mb"] = round(pte_mb, 3)
metrics["rule_h15_budget_mb"] = 30.0
metrics["rule_h15_compliance"] = "PASS" if pte_mb < 30.0 else "FAIL"
with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

final_leakage = json.load(open(ARTIFACT_DIR / "leakage_after.json"))
parity_obj = json.load(open(ARTIFACT_DIR / "parity_samples.json"))
assert Path(ARTIFACT_DIR / "vision.pte").exists(), "vision.pte missing"
assert Path(ARTIFACT_DIR / "labels.json").exists(), "labels.json missing"
assert Path(ARTIFACT_DIR / "thresholds.json").exists(), "thresholds.json missing"
assert Path(ARTIFACT_DIR / "class_counts_v2.json").exists(), "class_counts_v2.json missing"
if not final_leakage.get("zero_cluster_leakage"):
    raise RuntimeError("Leakage check failed.")
print("✓ All requested artifacts exist and verified.")
print("  Leakage zero:", final_leakage["zero_cluster_leakage"])
print("  Parity all pass:", parity_obj["all_pass"])
print("  PTE size MB:", round(pte_mb, 3))


In [ ]:
# Create a single ZIP containing the requested files and useful training extras.
DELIVERY_ZIP = RUN_DIR / "crop_doctor_training_deliverables.zip"
if DELIVERY_ZIP.exists():
    DELIVERY_ZIP.unlink()

required = [
    "metrics.json",
    "class_counts_v2.json",
    "thresholds.json",
    "vision.pte",
    "labels.json",
    "confusion_matrix.png",
    "confusion_matrix.csv",
    "parity_samples.json",
    "leakage_after.json",
    "split_v2.json",
    "probe_set.json",
]

with zipfile.ZipFile(DELIVERY_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in required:
        zf.write(ARTIFACT_DIR / name, arcname=name)
    # Add useful reproducibility files too.
    extras = ["best_model.pth", "training_history.json", "duplicate_report.json",
              "preprocessing_validation.json"]
    for name in extras:
        p = ARTIFACT_DIR / name
        if p.exists():
            zf.write(p, arcname=name)
print("Delivery ZIP:", DELIVERY_ZIP)
print("Size MB:", round(DELIVERY_ZIP.stat().st_size / (1024*1024), 2))


In [ ]:
# Download deliverables ZIP
if IS_COLAB:
    from google.colab import files
    files.download(str(DELIVERY_ZIP))
    print(f"✓ Download initiated for {DELIVERY_ZIP.name}")
else:
    print(f"\n✓ Training complete! Deliverables ZIP saved to:\n  {DELIVERY_ZIP}")
    print(f"\n👉 IN KAGGLE: Look at the right sidebar under 'Output' -> '/kaggle/working/runs_final/' -> click download on '{DELIVERY_ZIP.name}'!")


## Final files to send to Antigravity

The ZIP contains the 10 requested deliverables:

1. `metrics.json`
2. `class_counts_v2.json`
3. `thresholds.json`
4. `vision.pte`
5. `labels.json`
6. `confusion_matrix.png`
7. `parity_samples.json`
8. `leakage_after.json`
9. `split_v2.json`
10. `probe_set.json`

It also includes `best_model.pth`, `training_history.json`, `duplicate_report.json`, and `preprocessing_validation.json`.

**Do not send `vision.pte` for app integration as a verified model unless the parity test passes and the Android app uses a compatible ExecuTorch runtime version.** If the `.pte` export or parity cell fails, keep the PyTorch checkpoint and evaluation artifacts; the export issue must be fixed separately rather than claiming the `.pte` is ready.